# Computing your own correlation matrix

z-gain and thrivelines need an age-by-age **correlation matrix**: how strongly a person's z-score at one age is related to their z-score at another age.

Here we compute one from the LNM controls. A reliable matrix needs thousands of people scanned repeatedly. LNM has about 50 controls, so most of the matrix ends up extrapolated rather than estimated.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch, Rectangle

import pcntoolkit.util.output
from pcntoolkit import BLR, CorrelationMatrix, NormativeModel, load_fcon1000, load_lnm
from pcntoolkit.math_functions.basis_function import BsplineBasisFunction

# Hide PCNtoolkit's "Process: ..." progress messages
pcntoolkit.util.output.Output.set_show_messages(False)

REGION = "rh_G_front_middle_thickness"

## 1. z-scores for the controls

The matrix is built from z-scores, so we need a normative model first. We fit BLR on fcon1000, transfer it to the LNM site with 20% of the controls, and compute z-scores for the rest.

In [ ]:
fcon1000 = load_fcon1000().sel(response_vars=[REGION])
controls = load_lnm(group="control").sel(response_vars=[REGION])

fcon1000_model = NormativeModel(
    BLR(
        basis_function_mean=BsplineBasisFunction(degree=3, nknots=5),
        warp_name="warpsinharcsinh",
    ),
    inscaler="standardize",
    outscaler="standardize",
    savemodel=False,
    saveresults=False,
    saveplots=False,
)
fcon1000_model.fit(fcon1000)

# Randomly pick 20% of the controls for the adaptation group
control_subjects = controls.get_subject_ids()
all_controls = np.unique(control_subjects)
rng = np.random.default_rng(42)
adaptation_people = rng.choice(
    all_controls, size=int(0.2 * len(all_controls)), replace=False
)
in_adaptation_group = np.isin(control_subjects, adaptation_people)

# Adapt on visit 1 only; keep both visits of the others for the matrix
adapt_controls = controls.isel(
    observations=in_adaptation_group & (controls.get_visits() == 1)
)
reference_controls = controls.isel(observations=~in_adaptation_group)

lnm_model = fcon1000_model.transfer(adapt_controls)
lnm_model.predict(reference_controls);

## 2. Compute the matrix

`bandwidth` is the widest age gap, in years, that gets a correlation. We use 10, as in the precomputed matrix.

In [ ]:
corr = CorrelationMatrix.compute(reference_controls, bandwidth=10)

print(f"Subjects:       {corr.n_subjects}")
print(f"Observed ages:  {corr.estimated_range[0]} to {corr.estimated_range[1]}")

## 3. Estimated or extrapolated?

An age pair gets a correlation from the data only if at least 4 people were scanned at both ages. All other pairs are filled in by a regression on age and age gap, i.e. extrapolated.

In [ ]:
matrix = corr.matrix.sel(response_vars=REGION).values
ages = np.arange(matrix.shape[0])

# Count, for every pair of ages, the people scanned at both
df = reference_controls.to_dataframe()[["subject_ids", "X"]].droplevel(0, axis=1)
df["age"] = df["age"].round().astype(int)
n_people = np.zeros_like(matrix)
for first, second in df.groupby("subject_ids")["age"].agg(["min", "max"]).values:
    n_people[first, second] += 1
    n_people[second, first] += 1

age_gap = np.abs(ages[:, None] - ages[None, :])
in_bandwidth = (age_gap >= 1) & (age_gap <= corr.bandwidth)
estimated = in_bandwidth & (n_people >= 4)
n_estimated, n_total = estimated.sum() // 2, in_bandwidth.sum() // 2
print(f"Age pairs estimated from data: {n_estimated} of {n_total}")

In [ ]:
low, high = corr.estimated_range
# 1 = estimated, 0 = extrapolated, blank = beyond the bandwidth
support = np.where(in_bandwidth, estimated, np.nan)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
image = axes[0].imshow(matrix, origin="lower", cmap="coolwarm", vmin=-1, vmax=1)
fig.colorbar(image, ax=axes[0], label="correlation")
axes[0].set_title("Computed correlation matrix")

axes[1].imshow(
    support, origin="lower", cmap=ListedColormap(["#bdbdbd", "#c44e52"]), vmin=0, vmax=1
)
axes[1].set_title("Where each correlation comes from")
axes[1].legend(
    handles=[
        Patch(color="#c44e52", label="estimated (4+ people)"),
        Patch(color="#bdbdbd", label="extrapolated"),
        Patch(fill=False, ec="black", ls="--", label="observed ages"),
    ],
    loc="upper left",
)

for ax in axes:
    ax.add_patch(
        Rectangle((low, low), high - low, high - low, fill=False, ec="black", ls="--")
    )
    ax.set_xlabel("Age at scan B")
    ax.set_ylabel("Age at scan A")
    ax.grid(False)
plt.tight_layout()
plt.show()

Only a handful of age pairs have enough people. Everything else, including every age outside the observed range, is a regression guess, and the guesses are not plausible: z-scores a few years apart should not be negatively correlated.

Unless you have thousands of people scanned repeatedly, use a precomputed matrix (`load_correlation_matrix()`).

## 4. Save and load

`save()` writes one CSV per region to a folder; `load()` reads it back.

In [ ]:
corr.save("pcntoolkit_resources/data/my_correlation_matrix")
corr = CorrelationMatrix.load("pcntoolkit_resources/data/my_correlation_matrix")